# 图像卷积
:label:`sec_conv_layer`

上节我们解析了卷积层的原理，现在我们看看它的实际应用。由于卷积神经网络的设计是用于探索图像数据，本节我们将以图像为例。

## 互相关运算

严格来说，卷积层是个错误的叫法，因为它所表达的运算其实是*互相关运算*（cross-correlation），而不是卷积运算。
根据 :numref:`sec_why-conv`中的描述，在卷积层中，输入张量和核张量通过(**互相关运算**)产生输出张量。

首先，我们暂时忽略通道（第三维）这一情况，看看如何处理二维图像数据和隐藏表示。在 :numref:`fig_correlation`中，输入是高度为$3$、宽度为$3$的二维张量（即形状为$3 \times 3$）。卷积核的高度和宽度都是$2$，而卷积核窗口（或卷积窗口）的形状由内核的高度和宽度决定（即$2 \times 2$）。

![二维互相关运算。阴影部分是第一个输出元素，以及用于计算输出的输入张量元素和核张量元素：$0\times0+1\times1+3\times2+4\times3=19$.](../img/correlation.svg)
:label:`fig_correlation`

在二维互相关运算中，卷积窗口从输入张量的左上角开始，从左到右、从上到下滑动。
当卷积窗口滑动到新一个位置时，包含在该窗口中的部分张量与卷积核张量进行按元素相乘，得到的张量再求和得到一个单一的标量值，由此我们得出了这一位置的输出张量值。
在如上例子中，输出张量的四个元素由二维互相关运算得到，这个输出高度为$2$、宽度为$2$，如下所示：

$$
0\times0+1\times1+3\times2+4\times3=19,\\
1\times0+2\times1+4\times2+5\times3=25,\\
3\times0+4\times1+6\times2+7\times3=37,\\
4\times0+5\times1+7\times2+8\times3=43.
$$

注意，输出大小略小于输入大小。这是因为卷积核的宽度和高度大于1，
而卷积核只与图像中每个大小完全适合的位置进行互相关运算。
所以，输出大小等于输入大小$n_h \times n_w$减去卷积核大小$k_h \times k_w$，即：

$$(n_h-k_h+1) \times (n_w-k_w+1).$$

这是因为我们需要足够的空间在图像上“移动”卷积核。稍后，我们将看到如何通过在图像边界周围填充零来保证有足够的空间移动卷积核，从而保持输出大小不变。
接下来，我们在`corr2d`函数中实现如上过程，该函数接受输入张量`X`和卷积核张量`K`，并返回输出张量`Y`。


In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [10]:
def corr2d(X, K):  #@save
    """计算二维互相关运算"""
    h, w = K.shape               ##K means kernel, h means height, w means width
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))       ##initaslise Y with zeros, distribute the mempry of Y ,
                                                                        #which is determined by the shape of X and K
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()     ##* is element-wise multiplication, not matrix multiplication 
                                                           ##.sum() is the sum of all elements in the tensor
    return Y

通过 :numref:`fig_correlation`的输入张量`X`和卷积核张量`K`，我们来[**验证上述二维互相关运算的输出**]。


In [11]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

tensor([[19., 25.],
        [37., 43.]])

## 卷积层

卷积层对输入和卷积核权重进行互相关运算，并在添加标量偏置之后产生输出。
所以，卷积层中的两个被训练的参数是卷积核权重和标量偏置。
就像我们之前随机初始化全连接层一样，在训练基于卷积层的模型时，我们也随机初始化卷积核权重。

基于上面定义的`corr2d`函数[**实现二维卷积层**]。在`__init__`构造函数中，将`weight`和`bias`声明为两个模型参数。前向传播函数调用`corr2d`函数并添加偏置。


In [21]:
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))             #parameters of kernel initialized

    def forward(self, x):
        return corr2d(x, self.weight) + self.bias

In [14]:
##Notice that the kernel size is a tuple of two integers, which means that the kernel can be rectangular.
#Notice that the last row of code uses the broadcasting mechanism to add the bias term to the output of the convolution operation.
#corr2d(x, self.weight) returns a 2D tensor, and self.bias is a 1D tensor with a single element.

高度和宽度分别为$h$和$w$的卷积核可以被称为$h \times w$卷积或$h \times w$卷积核。
我们也将带有$h \times w$卷积核的卷积层称为$h \times w$卷积层。

## 图像中目标的边缘检测

如下是[**卷积层的一个简单应用：**]通过找到像素变化的位置，来(**检测图像中不同颜色的边缘**)。
首先，我们构造一个$6\times 8$像素的黑白图像。中间四列为黑色（$0$），其余像素为白色（$1$）。


In [15]:
X = torch.ones((6, 8))
X[:, 2:6] = 0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

接下来，我们构造一个高度为$1$、宽度为$2$的卷积核`K`。当进行互相关运算时，如果水平相邻的两元素相同，则输出为零，否则输出为非零。


In [16]:
K = torch.tensor([[1.0, -1.0]])

现在，我们对参数`X`（输入）和`K`（卷积核）执行互相关运算。
如下所示，[**输出`Y`中的1代表从白色到黑色的边缘，-1代表从黑色到白色的边缘**]，其他情况的输出为$0$。


In [17]:
Y = corr2d(X, K)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

现在我们将输入的二维图像转置，再进行如上的互相关运算。
其输出如下，之前检测到的垂直边缘消失了。
不出所料，这个[**卷积核`K`只可以检测垂直边缘**]，无法检测水平边缘。


In [18]:
corr2d(X.t(), K)

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

## 学习卷积核

如果我们只需寻找黑白边缘，那么以上`[1, -1]`的边缘检测器足以。然而，当有了更复杂数值的卷积核，或者连续的卷积层时，我们不可能手动设计滤波器。那么我们是否可以[**学习由`X`生成`Y`的卷积核**]呢？

现在让我们看看是否可以通过仅查看“输入-输出”对来学习由`X`生成`Y`的卷积核。
我们先构造一个卷积层，并将其卷积核初始化为随机张量。接下来，在每次迭代中，我们比较`Y`与卷积层输出的平方误差，然后计算梯度来更新卷积核。为了简单起见，我们在此使用内置的二维卷积层，并忽略偏置。


In [35]:
# 构造一个二维卷积层，它具有1个输出通道和形状为（1，2）的卷积核
conv2d = nn.Conv2d(1,1, kernel_size=(1, 2), bias=False)
##                (input_channels, out_channels, kernel_size, bias=False)
# channels 是输入通道数，out_channels 是输出通道数，kernel_size 是卷积核的大小，bias 是是否使用偏置项。
# 在这个例子中，输入通道数为1，输出通道数为1，卷积核大小为（1, 2），不使用偏置项。
#bias=False 是为教学演示服务的：把问题压缩到"只学核权重"，这样 10 轮迭代后学到的 [1, 0.97] 才能和手写的 [1, -1] 干净地对应上。
# 真实 CNN 里当然几乎总是开 bias（nn.Conv2d 默认就是 bias=True），因为真实任务中确实需要一个可学习的偏置来拟合数据。
# 初始化卷积核权重为0

# 这个二维卷积层使用四维输入和输出格式（批量大小、通道、高度、宽度），
# 其中批量大小和通道数都为1
X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))
lr = 3e-2  # 学习率

for i in range(20):   ##epochs
    Y_hat = conv2d(X)
    l = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    l.sum().backward()
    # 迭代卷积核
    conv2d.weight.data[:] -= lr * conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch {i+1}, loss {l.sum():.3f}')

epoch 2, loss 10.150
epoch 4, loss 2.175
epoch 6, loss 0.558
epoch 8, loss 0.173
epoch 10, loss 0.061
epoch 12, loss 0.024
epoch 14, loss 0.009
epoch 16, loss 0.004
epoch 18, loss 0.002
epoch 20, loss 0.001


在$10$次迭代之后，误差已经降到足够低。现在我们来看看我们[**所学的卷积核的权重张量**]。


In [36]:
conv2d.weight.data.reshape((1, 2))

tensor([[ 1.0024, -0.9972]])

以上cell的loss function理论根基



In [37]:
##l = (Y_hat - Y) ** 2



就是**平方误差**（这里用 `.sum()` 等价于 MSE，只是差一个常数倍），它背后的理论支撑主要有两条：

## 1. 最大似然估计（MLE）——最核心的理论依据

假设模型预测 $\hat{Y}$ 与真实目标 $Y$ 之间，误差服从**高斯（正态）分布**：

$$Y = \hat{Y} + \epsilon, \qquad \epsilon \sim \mathcal{N}(0, \sigma^2)$$

即 $Y \mid X \sim \mathcal{N}(\hat{Y}, \sigma^2)$。那么单个样本的概率密度是

$$p(Y \mid X, W) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\left(-\frac{(Y-\hat{Y})^2}{2\sigma^2}\right)$$

对整批数据取**负对数似然（NLL）**：

$$-\log p(\mathcal{D} \mid W) = \frac{1}{2\sigma^2}\sum_i (Y_i - \hat{Y}_i)^2 + \text{常数}$$

**关键结论**：$\sigma^2$ 和常数项都不影响参数 $W$ 的最优位置，所以

$$\boxed{\text{最小化平方误差} \iff \text{最大化高斯噪声假设下的似然}}$$

这正是 D2L 前面章节（线性回归）里讲过的：**"平方损失对应着误差服从高斯分布的假设"**。这个 cell 只是把这个结论直接用在卷积核学习上。

## 2. 最小二乘（Least Squares）/ 高斯-马尔可夫定理

- **最小二乘法**是最古老、最经典的估计准则：让残差平方和最小。
- **高斯-马尔可夫定理**（Gauss–Markov）：当误差满足"零均值、等方差、不相关"时，最小二乘估计是**所有线性无偏估计中方差最小的**（BLUE）。

所以平方损失不仅"方便求导"，还统计上有"最优线性无偏"的保证。

## 3. 为什么这里用回归损失而不是交叉熵

注意这个小节的任务是**回归**：
- 输入：$6\times8$ 图像
- 目标：$6\times7$ 的连续值输出 `Y`（值是 0、±1）
- 我们要学的是"输出 $Y$ 是由 $X$ 怎么卷积出来的" → 预测连续数值 → **回归问题** → 平方损失是自然选择。

而如果任务是分类（比如判断猫/狗），误差服从的是**类别分布**而不是高斯分布，那时对应的理论损失就是**交叉熵**——两者分别由"不同概率分布的 MLE"推导出来，是一个统一的框架。

## 两个小补充

**① 为什么用 `.sum()` 而不是 `.mean()`？**
两者最优解完全相同——因为 $\frac{1}{N}\sum e_i^2$ 只是 $\sum e_i^2$ 乘了个常数 $1/N$，梯度的方向（最优点的位置）不变。作者为了简单用了 `sum`。

**② 损失函数的"理论供给"正是训练能成功的原因**
正因为平方误差是一个**凸函数**（关于权重 $W$ 是二次的），梯度下降在这个单层问题上保证收敛到全局最优——这也是为什么 10 轮就能学到 `[1, -1]`。

---

一句话总结：**平方损失的理论支撑是"最大似然估计 + 高斯误差假设"**，对回归任务而言它既是统计上最优的准则，又是凸的可优化目标。这个 cell 就是这一理论在最简单场景下的实证。

细心的读者一定会发现，我们学习到的卷积核权重非常接近我们之前定义的卷积核`K`。

## 互相关和卷积

回想一下我们在 :numref:`sec_why-conv`中观察到的互相关和卷积运算之间的对应关系。
为了得到正式的*卷积*运算输出，我们需要执行 :eqref:`eq_2d-conv-discrete`中定义的严格卷积运算，而不是互相关运算。
幸运的是，它们差别不大，我们只需水平和垂直翻转二维卷积核张量，然后对输入张量执行*互相关*运算。

值得注意的是，由于卷积核是从数据中学习到的，因此无论这些层执行严格的卷积运算还是互相关运算，卷积层的输出都不会受到影响。
为了说明这一点，假设卷积层执行*互相关*运算并学习 :numref:`fig_correlation`中的卷积核，该卷积核在这里由矩阵$\mathbf{K}$表示。
假设其他条件不变，当这个层执行严格的*卷积*时，学习的卷积核$\mathbf{K}'$在水平和垂直翻转之后将与$\mathbf{K}$相同。
也就是说，当卷积层对 :numref:`fig_correlation`中的输入和$\mathbf{K}'$执行严格*卷积*运算时，将得到与互相关运算 :numref:`fig_correlation`中相同的输出。

为了与深度学习文献中的标准术语保持一致，我们将继续把“互相关运算”称为卷积运算，尽管严格地说，它们略有不同。
此外，对于卷积核张量上的权重，我们称其为*元素*。

## 特征映射和感受野

如在 :numref:`subsec_why-conv-channels`中所述， :numref:`fig_correlation`中输出的卷积层有时被称为*特征映射*（feature map），因为它可以被视为一个输入映射到下一层的空间维度的转换器。
在卷积神经网络中，对于某一层的任意元素$x$，其*感受野*（receptive field）是指在前向传播期间可能影响$x$计算的所有元素（来自所有先前层）。

请注意，感受野可能大于输入的实际大小。让我们用 :numref:`fig_correlation`为例来解释感受野：
给定$2 \times 2$卷积核，阴影输出元素值$19$的感受野是输入阴影部分的四个元素。
假设之前输出为$\mathbf{Y}$，其大小为$2 \times 2$，现在我们在其后附加一个卷积层，该卷积层以$\mathbf{Y}$为输入，输出单个元素$z$。
在这种情况下，$\mathbf{Y}$上的$z$的感受野包括$\mathbf{Y}$的所有四个元素，而输入的感受野包括最初所有九个输入元素。
因此，当一个特征图中的任意元素需要检测更广区域的输入特征时，我们可以构建一个更深的网络。

## 小结

* 二维卷积层的核心计算是二维互相关运算。最简单的形式是，对二维输入数据和卷积核执行互相关操作，然后添加一个偏置。
* 我们可以设计一个卷积核来检测图像的边缘。
* 我们可以从数据中学习卷积核的参数。
* 学习卷积核时，无论用严格卷积运算或互相关运算，卷积层的输出不会受太大影响。
* 当需要检测输入特征中更广区域时，我们可以构建一个更深的卷积网络。

## 练习

1. 构建一个具有对角线边缘的图像`X`。
    1. 如果将本节中举例的卷积核`K`应用于`X`，会发生什么情况？
    1. 如果转置`X`会发生什么？
    1. 如果转置`K`会发生什么？
1. 在我们创建的`Conv2D`自动求导时，有什么错误消息？
1. 如何通过改变输入张量和卷积核张量，将互相关运算表示为矩阵乘法？
1. 手工设计一些卷积核。
    1. 二阶导数的核的形式是什么？
    1. 积分的核的形式是什么？
    1. 得到$d$次导数的最小核的大小是多少？


In [38]:
# ===== 补充实验 1：验证「互相关 vs 严格卷积」输出等价 =====
# 结论：严格卷积(X, K) = 互相关(X, 翻转(K))；
#      若严格卷积学到的核 K' = 翻转(K)，则两者输出完全一致

X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])

# ① 互相关（框架实际实现：不翻转核）
cross_corr = corr2d(X, K)
print('互相关 corr2d(X, K)         =', cross_corr.tolist())

# ② 严格卷积 = 把核翻转 180° 后再做互相关
K_flip = torch.flip(K, dims=[0, 1])
print('严格卷积 corr2d(X, flip(K)) =', corr2d(X, K_flip).tolist())

# ③ 关键验证：若严格卷积学到的核 K' = flip(K)，
#    则 严格卷积(X, K') 应 == 互相关(X, K)
K_prime = torch.flip(K, dims=[0, 1])
out = corr2d(X, torch.flip(K_prime, dims=[0, 1]))
print("严格卷积(X, K'), 其中 K'=flip(K) =", out.tolist())
print('与互相关完全一致:', torch.equal(out, cross_corr))
print()

# ===== 补充实验 2：感受野随层数扩大（3x3 核，stride=1） =====
# 方法：逐一扰动每个输入像素，统计哪些会改变第 L 层输出[0,0]
# 预期：会改变的像素数 = (2L+1) x (2L+1)

def forward_layers(X, K, L):
    Y = X
    for _ in range(L):
        Y = corr2d(Y, K)
    return Y

X = torch.zeros((13, 13))   # 足够大，避免边界干扰
K = torch.ones((3, 3))      # 3x3 全 1 核

for L in range(1, 4):
    base = forward_layers(X, K, L)[0, 0].item()
    count = 0
    for p in range(13):
        for q in range(13):
            X2 = X.clone(); X2[p, q] = 1.0
            if abs(forward_layers(X2, K, L)[0, 0].item() - base) > 1e-6:
                count += 1
    rf = 2 * L + 1
    ok = '✓' if count == rf * rf else '✗'
    print(f'第 {L} 层：影响输出[0,0]的输入像素 = {count} 个，预期 {rf}x{rf}={rf*rf} 个  {ok}')

互相关 corr2d(X, K)         = [[19.0, 25.0], [37.0, 43.0]]
严格卷积 corr2d(X, flip(K)) = [[5.0, 11.0], [23.0, 29.0]]
严格卷积(X, K'), 其中 K'=flip(K) = [[19.0, 25.0], [37.0, 43.0]]
与互相关完全一致: True

第 1 层：影响输出[0,0]的输入像素 = 9 个，预期 3x3=9 个  ✓
第 2 层：影响输出[0,0]的输入像素 = 25 个，预期 5x5=25 个  ✓
第 3 层：影响输出[0,0]的输入像素 = 49 个，预期 7x7=49 个  ✓


这一部分是两个概念小节：**「互相关和卷积」**（澄清术语）和 **「特征映射和感受野」**（引入两个关键概念）。我分开讲透。

---

## 一、互相关和卷积

### 1. 两者的数学区别

| | 公式 | 核是否翻转 |
|---|---|---|
| **互相关**（cross-correlation） | $Y[i,j] = \sum_{a,b} K[a,b]\; X[i+a,\; j+b]$ | ❌ 不翻转，窗口直接加权求和 |
| **严格卷积**（convolution） | $Y[i,j] = \sum_{a,b} K[a,b]\; X[i-a,\; j-b]$ | ✅ 核要先**水平+垂直翻转 180°** |

回想上一节 `why-conv` 的公式：互相关用的是**和** $(i+a, j+b)$，而数学卷积定义用的是**差** $(i-a, j-b)$。差别就只在这一处。

### 2. 怎么互相转换（关键公式）

$$\text{严格卷积}(X, K) = \text{互相关}(X, \text{翻转}(K))$$

即：**把核翻转 180°，再做互相关，就得到严格卷积**。用代码说就是 `corr2d(X, torch.flip(K, dims=[0,1]))`。

### 3. 为什么"学出来的核"无所谓翻不翻转（本节核心论证）

课本的逻辑是：**核是数据学出来的，所以无论层实现哪种运算，输出都一样**。证明如下：

- 假设层做**互相关**，学到核 $K$，输出为 $Y[i,j]=\sum_{a,b}K[a,b]X[i+a,j+b]$
- 假设层做**严格卷积**，它会学到另一个核 $K'$。由转换公式，两者等价意味着 $K' = \text{翻转}(K)$
- 于是严格卷积的输出：
$$Y'[i,j]=\sum_{a,b}K'[a,b]X[i-a,j-b]=\sum_{a,b}K[-a,-b]X[i-a,j-b]=\sum_{a',b'}K[a',b']X[i+a',j+b']=Y[i,j]$$

（最后一步令 $a'=-a,\;b'=-b$。）

**结论**：两个系统学到的核互为翻转版，但对同一输入产生**完全相同的输出**。所以"翻转"这个差异会被学习过程自动吸收。

### 4. 一个具体例子

用 `fig_correlation` 的数据验证：



In [ ]:
X = [[0,1,2],[3,4,5],[6,7,8]],  K = [[0,1],[2,3]]
翻转 K:  K_flip = [[3,2],[1,0]]



- **互相关(X, K)** = `corr2d(X, K)` = **[[19,25],[37,43]]**
- **严格卷积(X, K)** = `corr2d(X, K_flip)` = [[5,11],[23,29]] ← 值不同！

但关键在于：如果做严格卷积的模型学到的是 $K'=\text{翻转}(K)$，即 $K'=[[3,2],[1,0]]$，那么**严格卷积(X, K')** = `corr2d(X, flip(K'))` = `corr2d(X, K)` = **[[19,25],[37,43]]** ✓

> 两个网络学到"长得不一样"的核，行为却一模一样。这就是"学习到的核翻转不影响输出"的实证。

### 5. 深度学习的惯例

> 文献里把"互相关运算"直接叫"卷积"，实际框架（PyTorch 的 `nn.Conv2d`）底层做的都是**不翻转核的互相关**。因为反正学出来的核会自动适配，翻转反而多此一举。

---

## 二、特征映射和感受野

### 1. 特征映射（feature map）——连接上一节的通道概念

上一节 `why-conv` 讲了"通道"，这里正式定义：

> 卷积层的**输出**就是特征映射（feature map）——它是把**输入映射到下一层空间维度的转换器**。

- 一个 $3\times3$ 输入经过核后得到 $2\times2$ 输出，这张 $2\times2$ 的输出图就是一张特征映射
- 有多个核就有多张特征映射（多个通道），每个通道记录"**某种特征在空间上哪里被激活**"

### 2. 感受野（receptive field）——本节最重要的新概念

> 某一层元素 $x$ 的**感受野** = 前向传播中**可能影响 $x$ 计算的所有元素**（来自所有先前层）。

一句话：**感受野 = 这个输出元素"看得见"的原始输入范围**。

### 3. 课本的两层例子（感受野如何随深度扩大）

**例 1**：输入 $3\times3$，核 $2\times2$ → 输出元素 `19` 的感受野 = 输入阴影区的 **4 个元素**（就是算它用的那 4 个）。

**例 2**：在 $2\times2$ 输出 $\mathbf{Y}$ 上**再叠一层**卷积，压成单个元素 $z$：
- $z$ 在 $\mathbf{Y}$ 上的感受野 = $\mathbf{Y}$ 全部 4 个元素
- $z$ 在**原始输入**上的感受野 = 最初全部 **9 个输入元素**

为什么是 9 个？因为 $\mathbf{Y}$ 的每个元素都"看"输入中的 $2\times2$，4 个元素合起来的范围正好覆盖整个 $3\times3$ 输入。

### 4. 感受野随深度增长的规律

堆叠 $L$ 层 $k\times k$ 卷积（stride=1）后，感受野尺寸为：

$$\text{感受野} = L\cdot(k-1) + 1$$

- $k=2, L=1$：$2$ → 感受野 $2\times2$
- $k=2, L=2$：$3$ → 感受野 $3\times3$
- $k=3, L=3$：$7$ → 感受野 $7\times7$

用图直观表示感受野如何"逐层扩散"：



In [ ]:
graph TD
    A["输入 X (3×3)"] --> B["第1层特征映射 (2×2)<br/>每个元素感受野 = 输入 2×2"]
    B --> C["第2层输出 z (1×1)<br/>z 在输入上的感受野 = 3×3 (全部9个元素)"]
    style A fill:#e8f0fe
    style B fill:#fef7e0
    style C fill:#e6f4ea



### 5. 为什么这决定了网络"要更深"

> **"当一个特征图中的任意元素需要检测更广区域的输入特征时，我们可以构建一个更深的网络。"**

因为：
- **底层**卷积核小、感受野小 → 只能看到局部（边缘、纹理）
- **深层**通过层层堆叠，每个元素看得越来越广 → 能组合局部特征为**物体部件、整体物体**
- 这正是 CNN 由浅入深提取"低级特征 → 高级特征"的根本机制，也是全书反复强调的直觉

---

## 一句话串联这一部分

> 互相关和严格卷积**只差核是否翻转**，但因为核是学出来的，两者完全等价（框架统一叫"卷积"）；卷积层输出即**特征映射**，而每个输出元素"看得见"的原始输入范围叫**感受野**，**堆叠越多层，感受野越大，能看到的模式越全局**——这就是 CNN 需要深度的原因。

---

需要我用一个小代码演示，把**两层卷积的感受野逐层可视化**出来，或者**验证互相关 vs 严格卷积输出等价**吗？

[Discussions](https://discuss.d2l.ai/t/1848)
